# H2S-NN Training Notebook

This notebook prepares the workspace and launches the new H2S-NN training entrypoint:

- `scripts/train_h2snn.py`
- `configs/experiments/acm_h2snn.yaml`
- `configs/experiments/dblp_h2snn.yaml`
- `configs/experiments/imdb_h2snn.yaml`

Recommended runtime: GPU.

In [6]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/H2S-NN_test")
else:
    PROJECT_DIR = Path.cwd()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}. Update PROJECT_DIR in this cell if needed."
    )

os.chdir(PROJECT_DIR)
print(f"Working directory: {PROJECT_DIR}")
print(f"Python executable: {sys.executable}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/H2S-NN_test
Python executable: /usr/bin/python3


In [7]:
%pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html


In [8]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE != "cuda":
    print("Warning: training will be much slower on CPU.")

Torch version: 2.6.0+cu124
Device: cuda


In [9]:
DATASET = "acm"  # one of: "acm", "dblp", "imdb"
CONFIG_BY_DATASET = {
    "acm": "configs/experiments/acm_h2snn.yaml",
    "dblp": "configs/experiments/dblp_h2snn.yaml",
    "imdb": "configs/experiments/imdb_h2snn.yaml",
}

CONFIG_PATH = CONFIG_BY_DATASET[DATASET]
EXP_NAME = f"{DATASET}_h2snn_notebook"
print(f"Config: {CONFIG_PATH}")
print(f"Experiment name: {EXP_NAME}")

Config: configs/experiments/acm_h2snn.yaml
Experiment name: acm_h2snn_notebook


In [ ]:
!python scripts/tune_h2snn.py \
  --config configs/experiments/acm_h2snn.yaml \
  --n_trials 30 --metric val_macro_f1

In [10]:
import subprocess
import sys

command = [
    sys.executable,
    "scripts/train_h2snn.py",
    "--config",
    CONFIG_PATH,
    "--exp_name",
    EXP_NAME,
    "--device",
    DEVICE,
]

print("Running:", " ".join(command))
subprocess.run(command, check=True)


Running: /usr/bin/python3 scripts/train_h2snn.py --config configs/experiments/acm_h2snn.yaml --exp_name acm_h2snn_notebook --device cuda


CalledProcessError: Command '['/usr/bin/python3', 'scripts/train_h2snn.py', '--config', 'configs/experiments/acm_h2snn.yaml', '--exp_name', 'acm_h2snn_notebook', '--device', 'cuda']' returned non-zero exit status 1.

In [17]:
import os
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "scripts/train_h2snn.py",
    "--config",
    CONFIG_PATH,
    "--exp_name",
    EXP_NAME,
    "--device",
    DEVICE,
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print("Running:", " ".join(command))
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")


Running: /usr/bin/python3 -u scripts/train_h2snn.py --config configs/experiments/acm_h2snn.yaml --exp_name acm_h2snn_notebook --device cuda
2026-04-07 12:45:59,725 | INFO | Dataset: acm
2026-04-07 12:45:59,726 | INFO | Target node type: paper
2026-04-07 12:45:59,726 | INFO | Metapaths: ['PAP', 'PSP']
2026-04-07 12:45:59,727 | INFO | Model parameters: 1589941
2026-04-07 12:46:06,548 | INFO | Epoch 0000 | train_loss: 1.1484 | train_acc: 0.3402 | train_macro_f1: 0.3059 | train_micro_f1: 0.3402 | val_loss: 1.1436 | val_acc: 0.3646 | val_macro_f1: 0.1781 | val_micro_f1: 0.3646 | test_acc: 0.3437 | test_macro_f1: 0.1732 | test_micro_f1: 0.3437
2026-04-07 12:46:06,808 | INFO | Checkpoint metric acc: 0.3646
2026-04-07 12:46:12,038 | INFO | Epoch 0001 | train_loss: 1.1183 | train_acc: 0.3802 | train_macro_f1: 0.2084 | train_micro_f1: 0.3802 | val_loss: 1.0792 | val_acc: 0.4420 | val_macro_f1: 0.4248 | val_micro_f1: 0.4420 | test_acc: 0.4792 | test_macro_f1: 0.4468 | test_micro_f1: 0.4792
2026-0

In [ ]:
from pathlib import Path
import json

exp_dir = Path("experiments") / EXP_NAME
print("Experiment directory:", exp_dir)
print("Files:")
for path in sorted(exp_dir.rglob("*")):
    print(path)

metrics_path = exp_dir / "metrics.jsonl"
if metrics_path.exists():
    print("\nLast metric rows:")
    lines = metrics_path.read_text(encoding="utf-8").strip().splitlines()
    for row in lines[-3:]:
        print(json.loads(row))

Experiment directory: experiments/acm_h2snn_notebook
Files:
experiments/acm_h2snn_notebook/config.yaml
